# Anonimización de clientes con pandas

El archivo `raw.csv` tiene datos personales de 600 clientes de un programa de fidelización.
El objetivo es publicar un archivo útil para análisis (gasto anual por segmento) sin que
se pueda identificar a ninguna persona.

Pasos:
1. Clasificar las columnas: identificadores directos, cuasi-identificadores y datos útiles.
2. Mostrar el riesgo: un atacante con información pública (`auxiliary.csv`) reidentifica clientes.
3. Anonimizar: suprimir, enmascarar, seudonimizar y generalizar.
4. Verificar que el ataque ya no funciona (k-anonimato).

In [1]:
import hashlib
import hmac

import pandas as pd

## 1. Carga y clasificación de columnas

In [2]:
raw = pd.read_csv("../data/raw.csv")
auxiliary = pd.read_csv("../data/auxiliary.csv")

print(raw.shape, auxiliary.shape)
raw.head()

(600, 8) (20, 4)


,name,document_id,email,loyalty_card_number,age,city,occupation,annual_spend
0,Camila González,1000000001,camila.gonzalez1@gmail.example,840000000001,25,Envigado,Comerciante,2580000
1,Mateo Ospina,1000000002,mateo.ospina2@yahoo.example,840000000002,36,Cali,Ingeniero de sistemas,5400000
2,Diana Duque,1000000003,diana.duque3@outlook.example,840000000003,38,Bogotá,Docente,5320000
3,Sebastián Álvarez,1000000004,sebastian.alvarez4@yahoo.example,840000000004,67,Itagüí,Abogada,4290000
4,Mariana Henao,1000000005,mariana.henao5@outlook.example,840000000005,29,Bogotá,Comerciante,3670000


| Tipo | Columnas | Tratamiento |
|---|---|---|
| Identificadores directos | `name`, `document_id`, `email` | Se eliminan |
| Identificador sensible que se necesita | `loyalty_card_number` | Se enmascara (solo últimos 4 dígitos) |
| Cuasi-identificadores | `age`, `city`, `occupation` | Se generalizan |
| Dato de análisis | `annual_spend` | Se conserva sin cambios |

Para poder unir registros sin revelar quién es cada cliente se agrega un seudónimo aleatorio `customer_id`.

## 2. Riesgo de reidentificación

`auxiliary.csv` simula información que un atacante puede conseguir (por ejemplo, redes sociales):
nombre, edad, ciudad y ocupación. Si la combinación edad + ciudad + ocupación es única en los
datos, el atacante sabe exactamente cuál registro es de esa persona.

In [3]:
quasi_identifiers = ["age", "city", "occupation"]

group_size = raw.groupby(quasi_identifiers)["annual_spend"].transform("size")
print(f"Registros únicos por edad + ciudad + ocupación: {(group_size == 1).sum()} de {len(raw)}")

Registros únicos por edad + ciudad + ocupación: 556 de 600


In [4]:
attack = auxiliary.merge(raw, on=quasi_identifiers, how="inner", suffixes=("_aux", "_raw"))
matches = attack.groupby("name_aux")["annual_spend"].count()
reidentified = matches[matches == 1].index

print(f"Personas del archivo auxiliar reidentificadas con certeza: {len(reidentified)} de {len(auxiliary)}")
attack[attack["name_aux"].isin(reidentified)][["name_aux", "age", "city", "occupation", "annual_spend"]].head()

Personas del archivo auxiliar reidentificadas con certeza: 20 de 20


,name_aux,age,city,occupation,annual_spend
0,Laura Gómez,47,Rionegro,Arquitecta,5840000
1,Andrés Restrepo,38,Envigado,Ingeniero de sistemas,5190000
2,Camila Henao,29,Pereira,Diseñadora gráfica,2460000
3,Carlos Muñoz,61,Bello,Técnico electricista,3370000
4,Natalia Ospina,34,Manizales,Docente,2710000


## 3. Anonimización

### 3.1 Generalización de cuasi-identificadores

In [5]:
def age_group(age):
    start = age // 10 * 10
    return f"{start}-{start + 9}"


REGIONS = {
    "Bello": "Andina",
    "Bogotá": "Andina",
    "Bucaramanga": "Andina",
    "Envigado": "Andina",
    "Itagüí": "Andina",
    "Manizales": "Andina",
    "Medellín": "Andina",
    "Pereira": "Andina",
    "Rionegro": "Andina",
    "Barranquilla": "Caribe",
    "Cartagena": "Caribe",
    "Cali": "Pacífica",
}

OCCUPATION_GROUPS = {
    "Comerciante": "Comercio y oficios",
    "Técnico electricista": "Comercio y oficios",
    "Docente": "Salud y educación",
    "Enfermera": "Salud y educación",
    "Médico": "Salud y educación",
    "Abogada": "Servicios profesionales",
    "Administradora": "Servicios profesionales",
    "Analista financiera": "Servicios profesionales",
    "Contadora": "Servicios profesionales",
    "Arquitecta": "Tecnología y diseño",
    "Diseñadora gráfica": "Tecnología y diseño",
    "Ingeniero de sistemas": "Tecnología y diseño",
}

unmapped = (set(raw["city"]) - set(REGIONS)) | (set(raw["occupation"]) - set(OCCUPATION_GROUPS))
assert not unmapped, f"Valores sin grupo: {unmapped}"

### 3.2 Enmascaramiento, seudónimos y supresión

- La tarjeta queda como `********` + últimos 4 dígitos (el número tiene 12 dígitos).
- `customer_id` es un HMAC-SHA256 del documento con una clave secreta: el mismo cliente
  recibe siempre el mismo seudónimo, pero sin la clave no se puede recalcular.

In [6]:
SECRET = b"clave-secreta-del-programa"


def masked_card(number):
    return "********" + str(number).zfill(12)[-4:]


def pseudonymize(document_id):
    digest = hmac.new(SECRET, str(document_id).encode(), hashlib.sha256).hexdigest()
    return f"CUST-{digest[:12].upper()}"


anonymized = pd.DataFrame(
    {
        "loyalty_card_number": raw["loyalty_card_number"].map(masked_card),
        "annual_spend": raw["annual_spend"],
        "customer_id": raw["document_id"].map(pseudonymize),
        "age_group": raw["age"].map(age_group),
        "region": raw["city"].map(REGIONS),
        "occupation_group": raw["occupation"].map(OCCUPATION_GROUPS),
    }
)

anonymized.head()

,loyalty_card_number,annual_spend,customer_id,age_group,region,occupation_group
0,********0001,2580000,CUST-EFF698B261D2,20-29,Andina,Comercio y oficios
1,********0002,5400000,CUST-E9F4EC0BD860,30-39,Pacífica,Tecnología y diseño
2,********0003,5320000,CUST-E2C7317583CA,30-39,Andina,Salud y educación
3,********0004,4290000,CUST-6AD4C8020C2B,60-69,Andina,Servicios profesionales
4,********0005,3670000,CUST-92569771F46C,20-29,Andina,Comercio y oficios


## 4. Verificación

**k-anonimato:** cada combinación de cuasi-identificadores debe aparecer al menos *k* veces.

In [7]:
generalized_qi = ["age_group", "region", "occupation_group"]
class_sizes = anonymized.groupby(generalized_qi).size()

print(f"Clases de equivalencia: {len(class_sizes)}")
print(f"k (tamaño de la clase más pequeña): {class_sizes.min()}")
class_sizes.describe()

Clases de equivalencia: 60
k (tamaño de la clase más pequeña): 1


count    60.000000
mean     10.000000
std      10.727756
min       1.000000
25%       3.000000
50%       4.500000
75%      19.250000
max      44.000000
dtype: float64

In [8]:
small = class_sizes[class_sizes < 3]
print(f"Registros en clases con menos de 3 personas: {small.sum()} de {len(anonymized)}")
small

Registros en clases con menos de 3 personas: 18 de 600


age_group  region    occupation_group   
20-29      Caribe    Tecnología y diseño    1
           Pacífica  Salud y educación      1
                     Tecnología y diseño    1
30-39      Caribe    Salud y educación      2
           Pacífica  Comercio y oficios     1
                     Tecnología y diseño    2
40-49      Caribe    Comercio y oficios     1
           Pacífica  Comercio y oficios     1
                     Tecnología y diseño    1
50-59      Pacífica  Comercio y oficios     2
                     Salud y educación      1
60-69      Caribe    Tecnología y diseño    2
           Pacífica  Comercio y oficios     1
                     Tecnología y diseño    1
dtype: int64

La generalización baja mucho el riesgo, pero algunas clases siguen siendo pequeñas (k = 1).
Los grupos de edad, región y ocupación los fija el formato de entrega, así que no se
generaliza más; para publicar estos datos de forma abierta habría que suprimir o agrupar
esas clases pequeñas.

El mismo ataque, ahora sobre los datos anonimizados: el atacante generaliza su información
de la misma forma y busca coincidencias.

In [9]:
auxiliary_generalized = pd.DataFrame(
    {
        "name": auxiliary["name"],
        "age_group": auxiliary["age"].map(age_group),
        "region": auxiliary["city"].map(REGIONS),
        "occupation_group": auxiliary["occupation"].map(OCCUPATION_GROUPS),
    }
)
attack_after = auxiliary_generalized.merge(anonymized, on=generalized_qi, how="inner")
candidates = attack_after.groupby("name").size()

print(f"Personas reidentificadas con certeza: {(candidates == 1).sum()} de {len(auxiliary)}")
print(f"Candidatos por persona (mínimo): {candidates.min()}")

Personas reidentificadas con certeza: 0 de 20
Candidatos por persona (mínimo): 3


In [10]:
assert len(anonymized) == len(raw)
assert anonymized["customer_id"].is_unique
assert anonymized["annual_spend"].tolist() == raw["annual_spend"].tolist()
assert not {"name", "document_id", "email", "age", "city", "occupation"} & set(anonymized.columns)

## 5. Utilidad de los datos anonimizados

El gasto anual sigue disponible por segmento.

In [11]:
anonymized.pivot_table(
    index="occupation_group", columns="region", values="annual_spend", aggfunc="mean"
).round(0)

region,Andina,Caribe,Pacífica
occupation_group,,,
Comercio y oficios,3462989.0,3456667.0,3006250.0
Salud y educación,3571190.0,3177500.0,3721818.0
Servicios profesionales,3966994.0,3231364.0,3660000.0
Tecnología y diseño,4001667.0,3895000.0,4246667.0


## 6. Almacenamiento

In [12]:
anonymized.to_csv("../submission/anonymized.csv", index=False)
pd.read_csv("../submission/anonymized.csv").head()

,loyalty_card_number,annual_spend,customer_id,age_group,region,occupation_group
0,********0001,2580000,CUST-EFF698B261D2,20-29,Andina,Comercio y oficios
1,********0002,5400000,CUST-E9F4EC0BD860,30-39,Pacífica,Tecnología y diseño
2,********0003,5320000,CUST-E2C7317583CA,30-39,Andina,Salud y educación
3,********0004,4290000,CUST-6AD4C8020C2B,60-69,Andina,Servicios profesionales
4,********0005,3670000,CUST-92569771F46C,20-29,Andina,Comercio y oficios
